In [ ]:
import os, importlib

REPO_URL = 'https://github.com/litcorp0/checkmaize.git'  # change if you fork the project

def repo_root():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            os.system(f'cd {repo} && git clean -fdq data/manifests')
            os.system(f'cd {repo} && git pull')
            return repo
        print('Repo not on this runtime yet. Cloning from GitHub...')
        result = os.system(f'git clone {REPO_URL} /content/checkmaize')
        if result != 0 or not os.path.exists(repo):
            print('Automatic clone failed. Likely causes:')
            print('  - the GitHub repo is private (make it public first), or')
            print('  - no internet on this runtime.')
            print('Manual fix - run this in a NEW cell, then re-run this cell:')
            print(f'  !git clone {REPO_URL} /content/checkmaize')
            print('Or drag the checkmaize folder into the Colab file explorer (into /content).')
            raise SystemExit
        return repo
    return os.path.abspath('..')

REPO = repo_root()
os.chdir(REPO)
print('Working in:', os.getcwd())

missing = []
for mod in ['numpy', 'PIL', 'pandas', 'yaml', 'sklearn', 'matplotlib', 'pytest']:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
if missing:
    print('installing missing packages:', missing)
    !pip install -q -r requirements.txt
    print('dependencies installed')
else:
    print('dependencies OK')

try:
    import torch
    print('torch:', torch.__version__, '| GPU available:', torch.cuda.is_available())
    if not torch.cuda.is_available():
        print('WARNING: no GPU on this runtime. In VS Code: click the Colab icon, disconnect,')
        print('connect again and choose a T4 GPU runtime. In browser Colab: Runtime ->')
        print('Change runtime type -> T4 GPU. Training on CPU takes many hours.')
except ImportError:
    print('torch is not installed in this environment. Install it (GPU build) before training.')


In [ ]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
!python -m pytest training/tests benchmarks/tests -v


In [ ]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
!python -m benchmarks.compare


In [ ]:
import os, shutil
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
shutil.make_archive('/content/report', 'zip', 'benchmarks/report')
shutil.make_archive('/content/runs', 'zip', 'artifacts/runs')
try:
    from google.colab import files
    files.download('/content/report.zip')
    files.download('/content/runs.zip')
    print('downloads started (browser Colab)')
except Exception:
    print('VS Code mode: the files are at /content/report.zip and /content/runs.zip')
    print('Drag them from the file explorer onto your computer.')
    print('Unzip report.zip into checkmaize/benchmarks/report/')
    print('Unzip runs.zip into checkmaize/artifacts/runs/')
